# Ablative Study - EfficientNet B2
Brain MRI classification trained on 3 datasets using EfficientNet-B2 (untrained weights).

## 1. Setup

In [1]:
#Install Dependencies
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 --upgrade --quiet
!pip install numpy pandas matplotlib seaborn scikit-learn torchmetrics grad-cam --quiet

In [2]:
import copy, time, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torchmetrics import Accuracy, Precision, Recall, F1Score
from sklearn.metrics import classification_report, confusion_matrix
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

try:
    from google.colab import drive
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# Device setup
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    torch.backends.cudnn.benchmark = True   # optimise conv for fixed input sizes
    torch.backends.cudnn.deterministic = False  # faster, fine for training
    gpu = torch.cuda.get_device_properties(0)
    print(f'Using GPU : {gpu.name}  ({gpu.total_memory / 1024**3:.1f} GB VRAM)')
else:
    DEVICE = torch.device('cpu')
    print('No CUDA GPU found — running on CPU')

print(f'PyTorch   : {torch.__version__}')
print(f'CUDA      : {torch.version.cuda}')
print(f'Running on: {"Google Colab" if ON_COLAB else "local environment"}')

Using GPU : NVIDIA GeForce RTX 4050 Laptop GPU  (6.0 GB VRAM)
PyTorch   : 2.10.0+cu126
CUDA      : 12.6
Running on: local environment


In [3]:
if ON_COLAB:
    drive.mount('/content/drive')
    OUTPUT_DIR = Path('/content/drive/MyDrive/COMP472/outputs/efficientnet_b2')
    preprocessed_dir = '/content/drive/MyDrive/brain_mri_preprocessed'
else:
    #  Local paths 
    OUTPUT_DIR       = Path('./outputs/efficientnet_b2')
    preprocessed_dir = './brain_mri_preprocessed'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output directory : {OUTPUT_DIR}')
print(f'Preprocessed data: {preprocessed_dir}')

Output directory : outputs\efficientnet_b2
Preprocessed data: ./brain_mri_preprocessed


## 2.Ablative Study Setups


In [ ]:
#Default , Base parameters
BATCH_SIZE    = 32
NUM_EPOCHS    = 20
LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 1e-4
IMG_SIZE      = 224

In [ ]:
# Experiment configurations for ablation study on Dataset 2
experiments = [
    {
        "name": "full_dataset",
        "classes": ["glioma", "meningioma", "notumor", "pituitary"],
        "images_per_class": None,
        "learning_rate": 0.001,
        "batch_size": 32
    },
    {
        "name": "reduced_classes",
        "classes": ["glioma", "meningioma", "pituitary"],
        "images_per_class": None,
        "learning_rate": 0.001,
        "batch_size": 32
    },
    {
        "name": "limited_images",
        "classes": ["glioma", "meningioma", "notumor", "pituitary"],
        "images_per_class": 50,
        "learning_rate": 0.001,
        "batch_size": 32
    },
    {
        "name": "very_limited_images",
        "classes": ["glioma", "meningioma", "notumor", "pituitary"],
        "images_per_class": 25,
        "learning_rate": 0.001,
        "batch_size": 32
    },
    {
        "name": "learning_rate_x10",
        "classes": ["glioma", "meningioma", "notumor", "pituitary"],
        "images_per_class": None,
        "learning_rate": 0.010,
        "batch_size": 32
    },
    {
        "name": "learning_rate_div2",
        "classes": ["glioma", "meningioma", "notumor", "pituitary"],
        "images_per_class": None,
        "learning_rate": 0.0005,
        "batch_size": 32
    },
    {
        "name": "batch_size_100",
        "classes": ["glioma", "meningioma", "notumor", "pituitary"],
        "images_per_class": None,
        "learning_rate": 0.001,
        "batch_size": 100
    },
    {
        "name": "batch_size_15",
        "classes": ["glioma", "meningioma", "notumor", "pituitary"],
        "images_per_class": None,
        "learning_rate": 0.001,
        "batch_size": 15
    },
]

print(f'{len(experiments)} experiments defined')
for exp in experiments:
    print(f'  {exp["name"]:<30} classes={len(exp["classes"])}  img/cls={exp["images_per_class"] or "all":>4}  lr={exp["learning_rate"]}  bs={exp["batch_size"]}')

## 3. Data

Images have been preprocessed to grayscale, tensors of 224×224 and normalized 0–255 → [0, 1].

### 3.1 Load Datasets

In [ ]:
from torch.utils.data import DataLoader

# Dataset 2 : 7,023 images — 4 classes
train_data_ds2 = torch.load(f'{preprocessed_dir}/dataset2_train.pt')
val_data_ds2   = torch.load(f'{preprocessed_dir}/dataset2_val.pt')
test_data_ds2  = torch.load(f'{preprocessed_dir}/dataset2_test.pt')

train_images_ds2   = train_data_ds2['images']
train_labels_ds2   = train_data_ds2['labels']
val_images_ds2     = val_data_ds2['images']
val_labels_ds2     = val_data_ds2['labels']
test_images_ds2    = test_data_ds2['images']
test_labels_ds2    = test_data_ds2['labels']
class_to_label_ds2 = train_data_ds2['class_to_label']   # e.g. {'glioma':0, 'meningioma':1, ...}

print(f'Dataset 2 loaded — class_to_label: {class_to_label_ds2}')
print(f'  Train: {len(train_images_ds2):,}  Val: {len(val_images_ds2):,}  Test: {len(test_images_ds2):,}')

### 3.2 Data Augmentation

MONAI transforms are applied on-the-fly during training only.

In [ ]:
!pip install monai --quiet

In [ ]:
import logging, os, sys, tempfile
from monai.transforms import (
    Compose,
    RandRotate,
    RandZoom,
    RandGaussianNoise,
    RandAdjustContrast,
    RandShiftIntensity,
)


class AugmentedDataset(torch.utils.data.Dataset):
    def __init__(self, images, labels, transform=None):
        self.images    = images
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label


def get_train_transforms():
    return Compose([
        RandRotate(range_x=0.15, prob=0.5, keep_size=True),
        RandZoom(min_zoom=0.95, max_zoom=1.05, prob=0.3, keep_size=True),
        RandGaussianNoise(mean=0.0, std=0.01, prob=0.2),
        RandAdjustContrast(gamma=(0.9, 1.1), prob=0.2),
        RandShiftIntensity(offsets=0.05, prob=0.2),
    ])

print('AugmentedDataset and transforms ready')

### 3.3 DataLoaders

In [ ]:
def build_exp_loaders(exp):
    """Filter Dataset 2 tensors for this experiment and return DataLoaders."""
    keep_classes     = exp['classes']
    images_per_class = exp['images_per_class']
    batch_size       = exp['batch_size']

    # Map each kept class name → its original label int, then remap to 0..N-1
    keep_label_ints = [class_to_label_ds2[c] for c in keep_classes]
    new_label_map   = {old: new for new, old in enumerate(keep_label_ints)}
    label_tensor    = torch.tensor(keep_label_ints)

    def filter_split(images, labels, limit_train=False):
        mask   = torch.isin(labels, label_tensor)
        imgs_f = images[mask]
        lbls_f = torch.tensor([new_label_map[l.item()] for l in labels[mask]], dtype=torch.long)

        if limit_train and images_per_class is not None:
            kept = []
            g    = torch.Generator(); g.manual_seed(SEED)
            for new_lbl in range(len(keep_classes)):
                idx = (lbls_f == new_lbl).nonzero(as_tuple=True)[0]
                idx = idx[torch.randperm(len(idx), generator=g)][:images_per_class]
                kept.append(idx)
            kept   = torch.cat(kept)
            imgs_f = imgs_f[kept]
            lbls_f = lbls_f[kept]

        return imgs_f, lbls_f

    tr_img, tr_lbl = filter_split(train_images_ds2, train_labels_ds2, limit_train=True)
    va_img, va_lbl = filter_split(val_images_ds2,   val_labels_ds2)
    te_img, te_lbl = filter_split(test_images_ds2,  test_labels_ds2)

    train_ds = AugmentedDataset(tr_img, tr_lbl, transform=get_train_transforms())
    val_ds   = AugmentedDataset(va_img, va_lbl, transform=None)
    test_ds  = AugmentedDataset(te_img, te_lbl, transform=None)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=0)

    print(f'  Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}  Classes: {keep_classes}')
    return train_loader, val_loader, test_loader, keep_classes

print('build_exp_loaders ready')

## 4. Model — EfficientNet B2

- Input channel modified: 3 to 1 (grayscale MRI)
- Classifier head replaced to match `num_classes` (4 or 44)
- Weights: random init (no pretrained weights)

In [ ]:
def build_efficientNet_b2(num_classes: int) -> nn.Module:
    model = models.efficientnet_b2(weights=None)
    # Replace first conv to accept 1-channel (grayscale) input
    model.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
    # Replace classifier head
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model.to(DEVICE)


# Sanity check
_m = build_efficientNet_b2(4)
_x = torch.randn(2, 1, IMG_SIZE, IMG_SIZE).to(DEVICE)
print(f'Output shape    : {list(_m(_x).shape)}')
print(f'Trainable params: {sum(p.numel() for p in _m.parameters() if p.requires_grad):,}')
print(_m.features[0][0])
del _m, _x

## 5. Training & Evaluation Functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, num_classes):
    model.train()
    loss_sum = 0.0
    acc_m = Accuracy(task='multiclass', num_classes=num_classes).to(DEVICE)
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * imgs.size(0)
        acc_m.update(out.argmax(1), labels)
    return loss_sum / len(loader.dataset), acc_m.compute().item()


@torch.no_grad()
def evaluate(model, loader, criterion, num_classes):
    model.eval()
    loss_sum = 0.0
    acc_m = Accuracy(task='multiclass', num_classes=num_classes).to(DEVICE)
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out = model(imgs)
        loss_sum += criterion(out, labels).item() * imgs.size(0)
        acc_m.update(out.argmax(1), labels)
    return loss_sum / len(loader.dataset), acc_m.compute().item()


@torch.no_grad()
def full_evaluation(model, loader, num_classes, class_names):
    model.eval()
    all_preds, all_labels = [], []
    ms = {
        'acc':  Accuracy( task='multiclass', num_classes=num_classes, average='macro').to(DEVICE),
        'prec': Precision(task='multiclass', num_classes=num_classes, average='macro').to(DEVICE),
        'rec':  Recall(   task='multiclass', num_classes=num_classes, average='macro').to(DEVICE),
        'f1':   F1Score(  task='multiclass', num_classes=num_classes, average='macro').to(DEVICE),
    }
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        preds = model(imgs).argmax(1)
        for m in ms.values(): m.update(preds, labels)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    results = {k: v.compute().item() for k, v in ms.items()}
    results.update({'all_preds': all_preds, 'all_labels': all_labels})

    print(f"  Accuracy : {results['acc']:.4f}")
    print(f"  Precision: {results['prec']:.4f}")
    print(f"  Recall   : {results['rec']:.4f}")
    print(f"  F1-Score : {results['f1']:.4f}")
    print(classification_report(all_labels, all_preds,
          target_names=(class_names if len(class_names) <= 20 else None), digits=4))
    return results


def train_model(model, cfg, save_path, learning_rate=LEARNING_RATE):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    history   = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val, best_w = float('inf'), None
    nc = cfg['num_classes']

    for ep in range(1, NUM_EPOCHS + 1):
        t0 = time.time()
        tl, ta = train_one_epoch(model, cfg['train_loader'], criterion, optimizer, nc)
        vl, va = evaluate(model, cfg['val_loader'], criterion, nc)
        scheduler.step(vl)
        for k, v in zip(history, [tl, vl, ta, va]): history[k].append(v)
        flag = ''
        if vl < best_val:
            best_val, best_w = vl, copy.deepcopy(model.state_dict())
            torch.save(best_w, save_path)
            flag = ' ✅'
        print(f'Ep [{ep:>2}/{NUM_EPOCHS}] Train {tl:.4f}/{ta:.4f} | Val {vl:.4f}/{va:.4f} | {time.time()-t0:.1f}s{flag}')

    model.load_state_dict(best_w)
    return model, history


print('Training & evaluation functions ready')

## 6. Visualization Functions

In [ ]:
def plot_curves(history, name, save_path):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(f'EfficientNet B2 — {name}', fontweight='bold')
    for ax, key, title in zip(axes, ['loss', 'acc'], ['Loss', 'Accuracy']):
        ax.plot(epochs, history[f'train_{key}'], 'b-o', ms=4, label='Train')
        ax.plot(epochs, history[f'val_{key}'],   'r-o', ms=4, label='Val')
        ax.set_title(title); ax.set_xlabel('Epoch')
        ax.legend(); ax.grid(alpha=0.3)
        if key == 'acc': ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.show()


def plot_confusion_matrix(results, class_names, name, save_path):
    cm = confusion_matrix(results['all_labels'], results['all_preds'], normalize='true')
    n  = len(class_names)
    fig, ax = plt.subplots(figsize=(min(n*0.7+2, 20), min(n*0.6+2, 18)))
    sns.heatmap(cm, annot=(n <= 15), fmt=('.2f' if n <= 15 else ''), cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f'Confusion Matrix — {name}', fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.xticks(rotation=45, ha='right', fontsize=(8 if n > 15 else 10))
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.show()


def plot_grad_cam(model, loader, class_names, name, save_path, n=8):
    model.eval()
    cam = GradCAM(model=model, target_layers=[model.features[-1]])
    imgs, labels = next(iter(loader))
    imgs = imgs[:n].to(DEVICE)
    with torch.no_grad():
        preds = model(imgs).argmax(1).cpu()
    mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
    fig, axes = plt.subplots(2, n // 2, figsize=(n * 2, 8))
    fig.suptitle(f'Grad-CAM — {name}', fontweight='bold')
    for i, ax in enumerate(axes.flatten()):
        gc = cam(input_tensor=imgs[i:i+1], targets=[ClassifierOutputTarget(preds[i].item())])[0]
        img_np = np.clip(std * imgs[i].cpu().numpy().transpose(1, 2, 0) + mean, 0, 1).astype(np.float32)
        ax.imshow(show_cam_on_image(img_np, gc, use_rgb=True))
        ax.set_title(f'True: {class_names[labels[i]]}\nPred: {class_names[preds[i]]}',
                     color=('green' if preds[i] == labels[i] else 'red'), fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.show()


print('Visualization functions ready')

## 7. Train on Dataset 2 - for each experiment

In [ ]:
all_results   = {}
all_histories = {}

for exp in experiments:
    exp_name    = exp['name']
    num_classes = len(exp['classes'])

    print(f'\n{"="*60}')
    print(f'  Experiment : {exp_name}')
    print(f'  Classes    : {exp["classes"]}  ({num_classes})')
    print(f'  Img/class  : {exp["images_per_class"] or "all"}')
    print(f'  LR         : {exp["learning_rate"]}   Batch: {exp["batch_size"]}')
    print(f'{"="*60}')

    train_loader, val_loader, test_loader, class_names = build_exp_loaders(exp)

    cfg = {
        'num_classes':   num_classes,
        'class_names':   class_names,
        'train_loader':  train_loader,
        'val_loader':    val_loader,
        'test_loader':   test_loader,
    }

    model = build_efficientNet_b2(num_classes)
    ckpt  = OUTPUT_DIR / f'efficientNet_b2_ds2_{exp_name}_best.pth'
    model, history = train_model(model, cfg, ckpt, learning_rate=exp['learning_rate'])

    #  Overfitting gap per epoch 
    print('\n── Overfitting Gap (Train Acc - Val Acc) per epoch ──')
    print(f'  {"Epoch":<8} {"Train Acc":>10} {"Val Acc":>10} {"Gap %":>8} {"Status":>12}')
    print(f'  {"-"*50}')
    for ep_idx, (tr, vl) in enumerate(zip(history['train_acc'], history['val_acc']), start=1):
        gap    = tr - vl
        status = '⚠ Overfit' if gap > 0.15 else ('⚠ Underfit' if tr < 0.60 and vl < 0.60 else '✓ OK')
        print(f'  {ep_idx:<8} {tr*100:>9.1f}% {vl*100:>9.1f}% {gap*100:>7.1f}% {status:>12}')
    print(f'\n  Final gap: {(history["train_acc"][-1] - history["val_acc"][-1])*100:.1f}%')

    #  Test evaluation 
    print('\n── Test Results ──')
    results = full_evaluation(model, cfg['test_loader'], num_classes, class_names)
    all_results[exp_name]   = results
    all_histories[exp_name] = history

    #  Val vs Test generalisation check 
    best_val_acc = max(history['val_acc'])
    print(f'\n── Generalisation Check ──')
    print(f'  Best Val Accuracy  : {best_val_acc*100:.2f}%')
    print(f'  Test Accuracy      : {results["acc"]*100:.2f}%')
    gap_vt = (best_val_acc - results["acc"]) * 100
    print(f'  Gap                : {gap_vt:.2f}%')
    if abs(gap_vt) > 5:
        print('  ⚠ Large gap between val and test — model may not generalise well')
    else:
        print('  ✓ Val and test accuracy are close — good generalisation')

    #  Grad-CAM 
    plot_grad_cam(model, cfg['test_loader'], class_names, exp_name,
                  OUTPUT_DIR / f'efficientNet_b2_ds2_{exp_name}_gradcam.png')

    del model; torch.cuda.empty_cache()

print('\nAll experiments done!')

In [ ]:
for exp in experiments:
    exp_name = exp['name']
    if exp_name not in all_results:
        continue

    history     = all_histories[exp_name]
    results     = all_results[exp_name]
    class_names = exp['classes']

    print(f'\n── Plots: {exp_name} ──')
    plot_curves(history, exp_name,
                OUTPUT_DIR / f'efficientNet_b2_ds2_{exp_name}_curves.png')
    plot_confusion_matrix(results, class_names, exp_name,
                OUTPUT_DIR / f'efficientNet_b2_ds2_{exp_name}_confusion.png')

    # Overfitting gap plot
    epochs = range(1, len(history['train_acc']) + 1)
    gaps   = [(tr - vl) * 100 for tr, vl in zip(history['train_acc'], history['val_acc'])]
    plt.figure(figsize=(8, 3))
    plt.plot(epochs, gaps, marker='o', color='crimson', linewidth=2)
    plt.axhline(y=15, color='orange', linestyle='--', label='Overfit threshold (15%)')
    plt.axhline(y=0,  color='green',  linestyle='--', label='Perfect generalization')
    plt.fill_between(epochs, gaps, 0, where=[g > 0 for g in gaps], alpha=0.15, color='crimson')
    plt.title(f'Overfitting Gap Over Epochs — {exp_name}')
    plt.xlabel('Epoch'); plt.ylabel('Train Acc − Val Acc (%)')
    plt.legend(); plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'efficientNet_b2_ds2_{exp_name}_gap.png', dpi=150)
    plt.show()

print('\nAll plots saved!')

## 8. Results Summary

In [ ]:
import json

exp_lookup = {exp['name']: exp for exp in experiments}

rows = [
    {
        'Experiment':       exp_name,
        'Model':            'EfficientNet B2 (untrained)',
        'Classes':          len(exp_lookup[exp_name]['classes']),
        'Class_Names':      ', '.join(exp_lookup[exp_name]['classes']),
        'Images_per_class': exp_lookup[exp_name]['images_per_class'] or 'all',
        'Batch_Size':       exp_lookup[exp_name]['batch_size'],
        'Epochs':           NUM_EPOCHS,
        'Learning_Rate':    exp_lookup[exp_name]['learning_rate'],
        'Weight_Decay':     WEIGHT_DECAY,
        'Image_Size':       IMG_SIZE,
        'Accuracy':         f"{r['acc']:.4f}",
        'Precision':        f"{r['prec']:.4f}",
        'Recall':           f"{r['rec']:.4f}",
        'F1-Score':         f"{r['f1']:.4f}",
    }
    for exp_name, r in all_results.items()
]

df = pd.DataFrame(rows)
print(df.to_string())
df.to_csv(OUTPUT_DIR / 'efficientnet_b2_ds2_ablation_results.csv', index=False)
print(f'\nSaved to {OUTPUT_DIR}/efficientnet_b2_ds2_ablation_results.csv')

# Save per-experiment training history
for exp_name, history in all_histories.items():
    history_path = OUTPUT_DIR / f'efficientNet_b2_ds2_{exp_name}_history.json'
    with open(history_path, 'w') as f:
        json.dump(history, f, indent=2)
    print(f'Saved history: {history_path.name}')